## 1. Setup dan lokasi data

In [ ]:
import glob, os
from pathlib import Path

import duckdb
import plotly.express as px
import polars as pl

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(80)

CANDIDATES = [
    "../data/raw/*.parquet", "data/raw/*.parquet",
    "../data/raw/data/*.parquet", "data/raw/data/*.parquet",
]
DATA_PATH = next((p for p in CANDIDATES if glob.glob(p)), None)
assert DATA_PATH, "File parquet tidak ketemu, cek folder data/raw/"

FILES = sorted(f.replace("\\", "/") for f in glob.glob(DATA_PATH))
ROOT = "../" if DATA_PATH.startswith("../") else ""
OUT_DIR = Path(ROOT + "output")
(OUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

total_bytes = sum(os.path.getsize(f) for f in FILES)
print(f"Path        : {DATA_PATH}")
print(f"Jumlah file : {len(FILES)}")
print(f"Total ukuran: {total_bytes / 1e6:,.0f} MB ({total_bytes / 1e9:.2f} GB)")


def collect(lf):
    try:
        return lf.collect(engine="streaming")
    except TypeError:
        return lf.collect(streaming=True)

## 2. Skema dan jumlah baris 

In [ ]:
lf = pl.scan_parquet(FILES)
schema = lf.collect_schema()
N_ROWS = lf.select(pl.len()).collect().item()

print(f"Baris : {N_ROWS:,}")
print(f"Kolom : {len(schema)}")

pl.DataFrame({
    "kolom": list(schema.names()),
    "tipe": [str(t) for t in schema.values()],
    "bersarang": [t.is_nested() for t in schema.values()],
})

In [ ]:
# baris per file, dibaca dari metadata jadi cepat
pl.DataFrame({
    "file": [os.path.basename(f) for f in FILES],
    "ukuran_mb": [round(os.path.getsize(f) / 1e6, 1) for f in FILES],
    "baris": [pl.scan_parquet(f).select(pl.len()).collect().item() for f in FILES],
})

## 3. Cuplikan data

In [ ]:
lf.head(5).collect()

## 4. SUMMARIZE dari DuckDB

Untuk ringkasan cepat per kolom (min, max, unik, null, dst). Kolom bersarang dilewati karena hasilnya tidak berguna, nanti dicek terpisah di bagian 8.

Supaya tidak lama, saya baru ringkas 3 file pertama. Kalau mau semuanya, ganti jadi `SUMMARIZE_FILES = FILES`.

In [ ]:
SUMMARIZE_FILES = FILES[:3]
MEMORY_LIMIT = "4GB"  

con = duckdb.connect()
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")


def q(sql):
    res = con.execute(sql)
    try:
        return res.pl()
    except Exception:
        rows = res.fetchall()
        return pl.DataFrame(rows, schema=[d[0] for d in res.description], orient="row")


def files_sql(files):
    return "[" + ", ".join(f"'{f}'" for f in files) + "]"


scalar_cols = [c for c, t in schema.items() if not t.is_nested()]
cols_sql = ", ".join(f'"{c}"' for c in scalar_cols)

summ = q(f"SUMMARIZE SELECT {cols_sql} FROM read_parquet({files_sql(SUMMARIZE_FILES)})")
summ.write_csv(OUT_DIR / "profiling_summarize.csv")
summ.select(["column_name", "column_type", "min", "max", "approx_unique", "avg", "null_percentage", "count"])